In [1]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params
from macromolecules.macromolecule import Macromolecule
from expression.build_me_model import flatten_list
from utils.parameters import human_model as m_model

No objective coefficients in model. Unclear what should be optimized


In [2]:
mu_val = 1e-9
n_cores = 10
counter = 7

lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

def add_boundary(m, type = 'sink', tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type =type)
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_' + type + '.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')
        

def _add_boundary(metabs_, type_ = 'sink', tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    if type(metabs_) != list:
        metabs_ = list(metabs_)

    for m in metabs_:
        if isinstance(m, cobra.Metabolite): # object
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type =type_)
        else: # string
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type =type_)
    
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat



In [3]:
# sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)
# tme1, sln1, stat1 = _add_sink(metabs_ = ['gdp_c'])

# tme_demand, sln_demand, stat_demand = _add_demand(metabs_ = [m.id for m in m_model.metabolites])

In [4]:
# import pandas as pd
# res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme0.reactions]})
# res_df['infeasible'] = res_df.reactions.apply(lambda x: sln0[tme0.reactions.index(x)])
# res_df.set_index(res_df.reactions, drop = True, inplace = True)
# res_df['feasible'] = pd.Series(res_df.index).apply(lambda r_id: sln_gdp_demand[tme_gdp_demand.reactions.index(r_id)]).tolist()
# res_df['feasible - infeasible'] = res_df.feasible - res_df.infeasible
# res_df['abs_diff'] = res_df['feasible - infeasible'].abs()
# res_df.sort_values(by = 'abs_diff', ascending = False, inplace = True)
# res_df.drop(columns = ['feasible - infeasible'], inplace = True)
# res_df['change'] = res_df[['feasible', 'infeasible']].apply(lambda x: 'increasing' if x[0] > x[1] else 'decreasing', axis = 1).tolist()


# # biom.sort_values(by = 'feasible', ascending = False, inplace = True)
# # biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

In [5]:
# tolerance = max([abs(v) for v in tme2.infeasible_reactions(mu_val = mu_val, sln = sln2, stat = stat2, tolerance = 0).values()])
# res = pd.DataFrame(columns = ['flux'])
# for r_id, flux in tme0.infeasible_reactions(mu_val = mu_val, sln = sln0, stat = stat0, tolerance = tolerance).items():
#     res.loc[r_id, 'flux'] = flux
# res['abs_flux'] = res.flux.abs()
# res.sort_values(by = 'abs_flux', ascending = False, inplace = True)
# test_metabs = list()
# for r_id in res.head(10).index:
#     test_metabs += [m.id for m in list(tme0.reactions.get_by_id(r_id).metabolites) if not hasattr(m, 'type')]
# test_metabs = sorted(set([r_id for r_id in test_metabs if 'biomass' not in r_id]))

# from itertools import product
# test = sorted(set(flatten_list([[m.id for m in tme0.reactions.get_by_id(r_id).metabolites if not isinstance(m, Macromolecule)] for r_id in flux_res.head(10).index])))
# test += [i[0] + i[1] for i in list(product(['a', 'c', 'u', 't'], ['mp_c', 'tp_c', 'dp_c']))]



In [6]:
# import multiprocessing
# import gc


# metabs_ = [m.id for m in m_model.metabolites]

# # demands
# print('Start demands')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_sink, zip(metabs_, ['demand']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')

# #sinks 
# # demands
# print('Start sinks')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_sink, zip(metabs_, ['sink']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')                       

In [7]:
sp = '/data2/hratch/human_me/other/test_lp/'
sinks = pd.read_csv(sp + 'test_sink.tab', sep = '\t')
demands = pd.read_csv(sp + 'test_demand.tab', sep = '\t')

sinks = sinks.sort_values(by = ['status', 'metabolite_id'], ascending = True).reset_index(drop = True)
demands = demands.sort_values(by = ['status', 'metabolite_id'], ascending = True).reset_index(drop = True)
boundary = pd.DataFrame(index = demands.metabolite_id)
boundary['demands'] = demands.status.tolist()

sinks.index = sinks.metabolite_id
sinks = sinks.loc[boundary.index, :]
boundary['sinks'] = sinks.status.tolist()

In [54]:
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)
# tme1, sln1, stat1 = _add_boundary(metabs_ = ['cdp_c'], type_ = 'sink')
tme1, sln1, stat1 = _add_boundary(metabs_ = ['cmp_c', 'gmp_c'], type_ = 'demand')

Getting MINOS parameters...
Done in 75.5286 seconds with status 1


/home/hratch/Projects/human_me/scripts/core/model.py:306 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 77.405 seconds with status 0


In [100]:
res = pd.DataFrame(index = [r.id for r in tme0.reactions])
for col, fm in {'infeasible': [sln0, tme0], 'DM': [sln1, tme1]}.items():
    res[col] = pd.Series(res.index).apply(lambda r_id: fm[0][fm[1].reactions.index(r_id)]).tolist()
# res['SK_diff'] = (res.SK - res.infeasible).abs()
res['DM_diff'] = (res.DM - res.infeasible).abs()
res.sort_values(by = ['DM_diff'], ascending = False, inplace = True)

In [96]:
tolerance = max(infeasible_reactions(self = tme1, mu_val = mu_val, 
                         sln = sln1, stat = stat1, tolerance = 0).values())

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/ipykernel_launcher.py:69 UserWarning: There is a discrepancy between the solver status and reactions that violate bound constraints


In [97]:
ir = infeasible_reactions(self = tme0, mu_val = mu_val, 
                         sln = sln0, stat = stat0, tolerance = tolerance)

In [104]:
res.loc[list(ir.keys()) + ['SRTNt6_2_r_R'], :]

,infeasible,DM,DM_diff
FATP7t_F_0,-1.067693e-20,0.000000e+00,1.067693e-20
FATP7t_R_0,-1.067693e-20,0.000000e+00,1.067693e-20
SRTNt6_2_r_R,-2.253167e-21,0.000000e+00,2.253167e-21
HGNC:14573_COPI_RETROtr,-5.272166e-21,0.000000e+00,5.272166e-21
HGNC:18265_TRANSLATION_ELONGATIONc,-3.749301e-21,0.000000e+00,3.749301e-21
HGNC:18265_post_TRANSLOC_3B_IMPORTtr,-3.749301e-21,-3.117116e-47,3.749301e-21
SRTNt6_2_r_R,-2.253167e-21,0.000000e+00,2.253167e-21


In [103]:
tme0.reactions.get_by_id('FATP7t_F_0').reaction

'1.0221896901172868e-06 HGNC:10996_enzyme_deg_proxy + 7.7054071914684e5*mu  1.02218969011729e6 HGNC:10996_folded_protein_pm + crvnc_c + na1_c --> crvnc_e + na1_e'

In [105]:
tme0.reactions.get_by_id('SRTNt6_2_r_R').reaction

'1.414118091371512e-06 HGNC:11050_enzyme_deg_proxy + 7.7095037001877e5*mu  1.41411809137151e6 HGNC:11050_folded_protein_pm + 2.0 k_e + 2.0 na1_c + srtn_c --> 2.0 k_c + 2.0 na1_e + srtn_e'

In [112]:
[r for r in tme0.reactions if 'HGNC:14573' in r.id and hasattr(r, 'cobra_id')][5].reaction

'2.207751907827065e-07 COPII_IMPORTtg_COMPLEX_enzyme_deg_proxy + 1.20362447848902e5*mu  2.20775190782706e7 COPII_IMPORTtg_complex_g + 2283579 HGNC:14573_folded_protein_r + 94 gtp_c + 94 h2o_c --> 2283579 HGNC:14573_folded_protein_g + 94 gdp_c + 94 h_c + 94 pi_c'

In [117]:
tme0.reactions.get_by_id('NDP7g').reaction

'1.4140957769777111e-06 HGNC:14573_enzyme_deg_proxy + 7.70938204634377e5*mu  1.41409577697771e6 HGNC:14573_folded_protein_g + h2o_g + udp_g --> h_g + pi_g + ump_g'

In [119]:
tme0.reactions.get_by_id('HGNC:14573_COPI_RETROtr').reaction

'2.3136917305759663e-07 COPI_RETROtr_COMPLEX_enzyme_deg_proxy + 1.31926421217663e5*mu  2.31369173057597e7 COPI_RETROtr_complex_g + 1224859 HGNC:14573_folded_protein_g + 127 gtp_c + 127 h2o_c --> 1224859 HGNC:14573_folded_protein_r + 127 gdp_c + 127 h_c + 127 pi_c'

In [144]:
[m for m in tme0.metabolites if hasattr(m, 'type') and m.type == 'complex'][0].decompose_complex()

{<Protein HGNC:4298_folded_protein_l at 0x7f1256f42ef0>: 1,
 <Protein HGNC:9251_folded_protein_l at 0x7f1256f42f28>: 1}

In [147]:
[m for m in tme0.metabolites if hasattr(m, 'type') and m.type == 'complex' and \
 ('HGNC:18265' in list([m.id for m in m.decompose_complex()]))]

[]

In [131]:
list(tme0.metabolites.get_by_id('HGNC:18265_folded_protein_l').reactions)[0].reaction

'2.8857087035463086e-07 Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_enzyme_deg_proxy + 1.57323366863067e5*mu  2.88570870354631e7 Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_g + 1241731 HGNC:18265_folded_protein_g + 44 gtp_c + 44 h2o_c --> 1241731 HGNC:18265_folded_protein_l + 44 gdp_c + 44 h_c + 44 pi_c'

In [138]:
tme0.metabolites.get_by_id('HGNC:18265_folded_protein_l')

Metabolite identifier,HGNC:18265_folded_protein_l
Name,
Memory address,0x07f1256139a58
Formula,C593H987N179O188S9
Compartment,l
In 1 reaction(s),HGNC:18265_Clathrin_IMPORTtl


In [156]:
from preprocess import parse_complex


In [164]:
hgnc_id = 'HGNC:18265'
m_model.genes.get_by_id(hgnc_id).reactions
complexes = parse_complex.eval_complex(m_model.reactions.get_by_id('ATPasel').gene_reaction_rule)

In [165]:
len(complexes)

192

In [166]:
complexes = [i for i in complexes if hgnc_id in i or hgnc_id == i]

In [167]:
len(complexes)

64

In [152]:
[r for r in tme0.reactions if 'ATPasel' in r.id]

[<ME_Reaction ATPasel_0 at 0x7f1256cc5f60>,
 <ME_Reaction ATPasel_10_COMPLEX_LYSOSOMAL_DEGRADATIONl at 0x7f1254596f60>,
 <Reaction ATPasel_10_COMPLEX_FORMATIONl at 0x7f12542a4c50>]

In [169]:
tme0.reactions.get_by_id('ATPasel_10_COMPLEX_FORMATIONl').metabolites

{<Protein HGNC:11647_folded_protein_l at 0x7f1256cc5fd0>: -1,
 <Protein HGNC:13527_folded_protein_l at 0x7f1256cd3048>: -1,
 <Protein HGNC:13724_folded_protein_l at 0x7f1256cd30b8>: -1,
 <Protein HGNC:16832_folded_protein_l at 0x7f1256cd3128>: -1,
 <Protein HGNC:18125_folded_protein_l at 0x7f1256cd3198>: -1,
 <Protein HGNC:18303_folded_protein_l at 0x7f1256cd3208>: -1,
 <Protein HGNC:851_folded_protein_l at 0x7f1256cd32b0>: -1,
 <Protein HGNC:854_folded_protein_l at 0x7f1256cd32e8>: -1,
 <Protein HGNC:855_folded_protein_l at 0x7f1256cd3358>: -1,
 <Protein HGNC:856_folded_protein_l at 0x7f1256cd33c8>: -1,
 <Protein HGNC:861_folded_protein_l at 0x7f1256cd3438>: -1,
 <Protein HGNC:862_folded_protein_l at 0x7f1256cd34a8>: -1,
 <Protein HGNC:863_folded_protein_l at 0x7f1256cd3518>: -1,
 <Complex ATPasel_10_complex_l at 0x7f1256cc5f98>: 1}